# RAG Evaluation

Dieses Notebook fuehrt die RAG-Pipeline fuer alle Evaluationsfragen aus,
speichert die Ergebnisse (Antworten + Chunks) als JSON und erzeugt
einen Markdown-Report zur manuellen Bewertung.

### Kategorien im Fragenkatalog
- **A** – Faktische Detail- und Datenfragen
- **B** – Erklaerungsfragen (Mechanismen, Gruende)
- **C** – Vergleichs- und Mehrquellenfragen
- **D** – Uneindeutige oder nicht beantwortbare Fragen


In [1]:
# Setup: Projekt-Root setzen
import os
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)
print('ROOT:', ROOT)



ROOT: c:\Users\Mauricio.Guerrero\Documents\rag_project


In [7]:
# Konfiguration

# --- Modell-Auswahl ---
# 'llama' → llama3.1:8b  (Evaluation 1)
# 'mistral' → mistral-nemo (Evaluation 2)
MODEL_CHOICE = 'llama'

import src.config as cfg
if MODEL_CHOICE == 'mistral':
    cfg.DEFAULT_LLM_MODEL = 'mistral-nemo'
    cfg.PROMPT_VERSION = 'mistral'
else:
    cfg.DEFAULT_LLM_MODEL = 'llama3.1:8b'
    cfg.PROMPT_VERSION = 'llama'

# --- RAG-Einstellungen ---
RAG_MODE       = 'hybrid'
RAG_K          = 20
CHUNK_SIZE     = 1200
CHUNK_OVERLAP  = 200
USE_RERANKER   = True
RERANK_TOP_N   = 5

# Welche Fragen evaluieren? None = alle
QUESTION_IDS = None

print(f'Modell: {cfg.DEFAULT_LLM_MODEL} | Prompt: {cfg.PROMPT_VERSION}')
print(f'RAG-Modus: {RAG_MODE} | k={RAG_K} | Reranker={USE_RERANKER} | top_n={RERANK_TOP_N}')


In [8]:
# Fragen laden
from src.evaluation import load_questions

QUESTIONS_PATH = ROOT / 'data' / 'eval' / 'questions.jsonl'
all_questions = load_questions(QUESTIONS_PATH)

if QUESTION_IDS is not None:
    questions = [q for q in all_questions if q['id'] in QUESTION_IDS]
else:
    questions = all_questions

print(f'{len(questions)} Fragen geladen.')
for q in questions:
    print(f"  [{q['id']}] Kategorie {q['category']}: {q['query'][:90]}...")

print('Lese von:', QUESTIONS_PATH)
print('Datei existiert:', QUESTIONS_PATH.exists())


37 Fragen geladen.
  [q1] Kategorie A: Welche Füll- und Lösezeiten gelten für den Bremszylinder in der Bremsstellung G und P nach...
  [q2] Kategorie A: Welche Komponenten verursachten laut Peche die meisten außerplanmäßigen Instandhaltungsmaß...
  [q3] Kategorie A: Wie viele Maßnahmenvorschläge und Handlungsfelder wurden im DZSF-Bericht identifiziert?...
  [q4] Kategorie A: Welche Sensitivitätsindizes nach Sobol' werden in der Dissertation von Jobstfinke verwende...
  [q5] Kategorie A: Welche relevanten Normen werden bei der Entwicklung der automatisierten Bremsprobe nach Pe...
  [q6] Kategorie A: Welche drei Entwicklungen im Eisenbahnwesen nennt Jobstfinke als potenzielle Einflussfakto...
  [q7] Kategorie A: Welche Zuglänge in Metern und welche Art der Bremsung werden im Prüfszenario 4 zur Plausib...
  [q8] Kategorie A: Wie ist der Sensitivitätsindex erster Ordnung Si nach Sobol' mathematisch definiert und wa...
  [q9] Kategorie B: Warum reicht der Sensitivitätsindex erster Ordnung a

In [ ]:
# RAG fuer alle Fragen ausfuehren und Ergebnisse sammeln
# Hinweis: Dieser Schritt laeuft lokal (Ollama + Reranker) und dauert je nach
# Hardware ca. 30-120 Sekunden pro Frage.
import time
from src.evaluation import run_rag_for_eval

rag_results = []

for i, q in enumerate(questions, start=1):
    print(f'[{i}/{len(questions)}] {q["id"]} ({q["category"]}): {q["query"][:60]}...')
    t0 = time.perf_counter()

    result = run_rag_for_eval(
        query=q['query'],
        mode=RAG_MODE,
        k=RAG_K,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        use_reranker=USE_RERANKER,
        rerank_top_n=RERANK_TOP_N,
        llm_model=cfg.DEFAULT_LLM_MODEL,
    )

    dt = time.perf_counter() - t0
    rag_results.append({
        'id': q['id'],
        'category': q['category'],
        'query': q['query'],
        'expected_output': q.get('expected_output', ''),
        'actual_output': result['answer'],
        'retrieval_context': result['retrieval_context'],
        'sources': result['sources'],
        'latency_s': round(dt, 2),
    })
    print(f'  -> {dt:.1f}s | Antwort: {result["answer"][:100]}...')

print(f'Alle {len(rag_results)} RAG-Laeufe abgeschlossen.')


In [10]:
# RAG-Ergebnisse als JSON speichern (Antworten + Chunks fuer manuelle Bewertung)

import json

eval_dir = ROOT / 'data' / 'eval'
eval_dir.mkdir(parents=True, exist_ok=True)

rag_only = []
for r in rag_results:
    chunks_with_source = []
    for i, chunk_text in enumerate(r['retrieval_context']):
        src = r['sources'][i] if i < len(r['sources']) else {}
        chunks_with_source.append({
            'rank': i + 1,
            'text': chunk_text,
            'source': src.get('source') or '–',
            'page': src.get('page'),
        })
    rag_only.append({
        'id': r['id'],
        'category': r['category'],
        'query': r['query'],
        'expected_output': r.get('expected_output', ''),
        'actual_output': r['actual_output'],
        'latency_s': r['latency_s'],
        'top_chunks': chunks_with_source,
        'metrics': [],  # noch keine Metriken
    })

rag_path = eval_dir / 'results_full.json'
with open(rag_path, 'w', encoding='utf-8') as f:
    json.dump(rag_only, f, ensure_ascii=False, indent=2)

print(f'{len(rag_only)} RAG-Ergebnisse gespeichert: {rag_path}')
print('Jetzt kannst du die Markdown-Zelle ausführen um Antworten + Chunks zu sehen.')


37 RAG-Ergebnisse gespeichert: c:\Users\Mauricio.Guerrero\Documents\rag_project\data\eval\results_full.json
Jetzt kannst du die Markdown-Zelle ausführen um Antworten + Chunks zu sehen.


In [ ]:
# Markdown-Report aus results_full.json erzeugen
import json
from pathlib import Path

eval_dir = ROOT / 'data' / 'eval'
json_path = eval_dir / 'results_full.json'

with open(json_path, 'r', encoding='utf-8') as f:
    full_results = json.load(f)

lines = []
lines.append('# RAG Evaluation Report')
lines.append(f'**Anzahl Fragen:** {len(full_results)}  ')
lines.append(f'**RAG-Modus:** {RAG_MODE} | k={RAG_K} | Reranker={USE_RERANKER} | top_n={RERANK_TOP_N}  ')
lines.append('')
lines.append('---')

for entry in full_results:
    lines.append('')
    lines.append(f"## [{entry['id']}] Kategorie {entry['category']}")
    lines.append(f"**Frage:** {entry['query']}")
    lines.append('')
    lines.append('**Generierte Antwort:**')
    lines.append(f"> {entry['actual_output'].strip()}")

    expected = entry.get('expected_output', '').strip()
    if expected:
        lines.append('')
        lines.append('**Erwartete Antwort:**')
        lines.append(f'> {expected}')

    top_chunks = entry.get('top_chunks', [])[:5]
    if top_chunks:
        lines.append('')
        lines.append('**Top Chunks (Retrieval Context):**')
        lines.append('')
        for chunk in top_chunks:
            src = chunk.get('source') or '-'
            page = chunk.get('page')
            src_label = Path(src).name if src != '-' else '-'
            page_label = f', Seite {page}' if page is not None else ''
            summary = f"Chunk {chunk['rank']} — {src_label}{page_label}"
            lines.append(f'<details><summary>{summary}</summary>')
            lines.append('')
            lines.append('```')
            lines.append(chunk['text'].strip())
            lines.append('```')
            lines.append('')
            lines.append('</details>')
            lines.append('')

    lines.append('---')

md_path = eval_dir / 'results_report.md'
with open(md_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines) + '\n')

print(f'Markdown-Report gespeichert: {md_path}')